<a href="https://colab.research.google.com/github/nembrinj/ARCHEST_2026/blob/main/colab/3DGaussianSplatting_from_INRIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3D Gaussian splatting from INRIA

This colab notebook installs gaussian-splatting software with all requirements (including appropriate cuda toolkit versions) and starts a training process. It requires GPU compute time.

Intallation time is approx 21-22 minutes (with GPU).

Then the instance can be used for training gaussian splatting representations. Note that data upload time is not insignificant


# Python downgrading

This is necessary to use the specific repo which was built using python 3.7

This cell takes less than 30sec to compute


In [1]:
!wget -O mini.sh https://repo.anaconda.com/miniconda/Miniconda3-py37_23.1.0-1-Linux-x86_64.sh
!chmod +x mini.sh
!bash ./mini.sh -b -f -p /usr/local
!conda install -q -y python=3.7
import sys
sys.path.append('/usr/local/lib/python3.7/site-packages')
!python --version  # Should say Python 3.7.x

--2026-08-16 09:44:36--  https://repo.anaconda.com/miniconda/Miniconda3-py37_23.1.0-1-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.191.158, 104.16.32.241, 2606:4700::6810:bf9e, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.191.158|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 90665082 (86M) [application/x-sh]
Saving to: ‘mini.sh’

mini.sh             100%[===================>]  86.46M   220MB/s    in 0.4s    

2026-08-16 09:44:37 (220 MB/s) - ‘mini.sh’ saved [90665082/90665082]

PREFIX=/usr/local
Unpacking payload ...
                                                                                              
Installing base environment...





Preparing transaction: - \ | done
Executing transaction: - \ | / - \ | / - \ | / - \ | / - \ done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when ru

# CUDA 11.8

This takes approx. 14 min to compute

In [3]:
# check if cuda installer already downloaded
!ls -al

total 385532
drwxr-xr-x 1 root root      4096 Aug 16 09:45 .
drwxr-xr-x 1 root root      4096 Aug 16 09:37 ..
drwxr-xr-x 4 root root      4096 Aug 10 13:31 .config
-rwxr-xr-x 1 root root 304087040 Aug 16 09:45 cuda_11.8.0_520.61.05_linux.run
drwx------ 6 root root      4096 Aug 16 09:43 drive
drwxr-xr-x 3 root root      4096 Aug 16 09:43 gaussian-splatting
-rwxr-xr-x 1 root root  90665082 Feb 13  2025 mini.sh
drwxr-xr-x 1 root root      4096 Aug 10 13:31 sample_data


In [ ]:
# download only if needed (this is the main time consuming part)
!wget https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run
!chmod +x cuda_11.8.0_520.61.05_linux.run

In [4]:
# install appropriate cuda version
!./cuda_11.8.0_520.61.05_linux.run --silent --toolkit --no-drm --no-man-page
import os
os.environ['PATH'] += ':/usr/local/cuda-11.8/bin'
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.8/lib64:/usr/lib64-nvidia'
!nvcc --version  # Should show CUDA 11.8


gzip: stdin: unexpected end of file
./cuda_11.8.0_520.61.05_linux.run: 226: cannot create /dev/tty: No such device or address
./cuda_11.8.0_520.61.05_linux.run: 226: cannot create /dev/tty: No such device or address
Signal caught, cleaning up
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


#Pytorch with cuda

This cell takes approx 2-3 min to compute.

__As it requires session restart, do not run cells further before complete__

In [3]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch==1.12.1+cu116 torchvision==0.13.1+cu116 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu116

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu116
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB ? eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 26.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 17.4 MB/s eta 0:00:00
  Obtaining dependency information for pillow!=8.3.*,>=5.3.0 from https://files.pythonhosted.org/packages/2c/a2/2d565cb1d754384a88998b9c86daf803a3a7908577875231eb99b8c7973d/Pillow-9.5.0-cp37-cp37m-manylinux_2_28_x86_64.whl.metadata
Discarding https://files.pythonhosted.org/packages/2c/a2/2d565cb1d754384a88998b9c86daf803a3a7908577875231eb99b8c7973d/Pillow-9.5.0-cp37-cp37m-manylinux_2_28_x86_64.whl#sha256=35f6e77122a0c0762268216315bf239cf52b88865bba522999dc38f1c52b9b47 (from https://download.pytorch.org/whl/cu116/pillow/) (requires-python:>=3.7): Requested pillow!=8.3.*,>=5.3.0 from https://files.pythonhosted.org/packages/2c/a2/2d565cb1d754384a88998b9c86daf803a3a7908577875231eb9

# Verification of the versions

In [1]:
import torch
print(torch.cuda.is_available())  # Should be True
print(torch.version.cuda)        # Should be 11.3 (from PyTorch)
print(torch.cuda.get_device_name(0))  # Should show GPU

True
12.8
Tesla T4


In [2]:
!nvidia-smi

Sun Aug 16 09:43:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# clone the INRIA gaussian-splatting git and intall it
%cd /content
!git clone --recursive https://github.com/camenduru/gaussian-splatting
!pip install -q plyfile

%cd /content/gaussian-splatting
!pip install -q /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q /content/gaussian-splatting/submodules/simple-knn


/content
fatal: destination path 'gaussian-splatting' already exists and is not an empty directory.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.0 MB/s eta 0:00:00
/content/gaussian-splatting
ERROR: Invalid requirement: '/content/gaussian-splatting/submodules/diff-gaussian-rasterization': Expected package name at the start of dependency specifier
    /content/gaussian-splatting/submodules/diff-gaussian-rasterization
    ^
Hint: It looks like a path. File '/content/gaussian-splatting/submodules/diff-gaussian-rasterization' does not exist.
ERROR: Invalid requirement: '/content/gaussian-splatting/submodules/simple-knn': Expected package name at the start of dependency specifier
    /content/gaussian-splatting/submodules/simple-knn
    ^
Hint: It looks like a path. File '/content/gaussian-splatting/submodules/simple-knn' does not exist.


In [3]:
# upload your data
# 1. zip your data on the server
# 2. download it locally, put it on your google drive (new->file upload...)
# 3. run the cell to select the file and unzip it

from google.colab import drive

drive.mount('/content/drive')

# if your zip file is in My Drive/colmap/
data_path = "/content/drive/MyDrive/colmap"

# change to the name of the zip file
zip_name="test"

dir = "/content/gaussian-splatting/"+zip_name
zipfile = data_path+"/"+zip_name+".zip"

!mkdir -p $dir
!unzip -q $zipfile -d $dir


Mounted at /content/drive


In [5]:
# train the gaussian splats
!python train.py -s /content/gaussian-splatting/test

python3: can't open file '/content/train.py': [Errno 2] No such file or directory


In [ ]:
# use a standard dataset for testing
!wget https://huggingface.co/camenduru/gaussian-splatting/resolve/main/tandt_db.zip
!unzip tandt_db.zip

!python train.py -s /content/gaussian-splatting/tandt/train